# Audio Flamingo 3 (NVIDIA)

In [ ]:
import tempfile
from pathlib import Path

import librosa
import soundfile as sf
import pandas as pd
from tqdm import tqdm
from sklearn.metrics import accuracy_score, f1_score
from transformers import AudioFlamingo3ForConditionalGeneration, AutoProcessor

In [ ]:
!source /etc/network_turbo

In [ ]:
import sys
from pathlib import Path

for _base in (Path.cwd(), *Path.cwd().parents):
    _llm_dir = _base if (_base / "llm_utils").is_dir() else _base / "ad_detection" / "train_notebook" / "LLM"
    if (_llm_dir / "llm_utils").is_dir():
        if str(_llm_dir) not in sys.path:
            sys.path.insert(0, str(_llm_dir))
        break

from llm_utils.config import PROJECT_ROOT, CACHE_DIR
MODEL_ID     = "nvidia/audio-flamingo-3-hf"


processor = AutoProcessor.from_pretrained(MODEL_ID, cache_dir=CACHE_DIR)
model = AudioFlamingo3ForConditionalGeneration.from_pretrained(
    MODEL_ID,
    torch_dtype="auto",
    device_map="auto",
    cache_dir=CACHE_DIR,
)

print("Audio Flamingo 3 model loaded.")

In [ ]:
SYSTEM_PROMPT = (
    "You are a clinical speech-language pathologist specialized in detecting "
    "Alzheimer's disease and dementia from spontaneous speech. You analyze speech "
    "patterns including: word-finding difficulties, semantic paraphasias, empty speech, "
    "reduced syntactic complexity, repetitions, incomplete utterances, and pragmatic "
    "impairments. Based on the audio, classify the speaker."
)

USER_PROMPT = (
    "Listen to this speech sample carefully. Based on the speech characteristics, "
    "is this speaker showing signs of dementia or is this a healthy control? "
    "Answer with exactly one word: 'Dementia' or 'Control'."
)

In [ ]:
TMP_WAV_DIR = Path(tempfile.mkdtemp(prefix="flamingo3_wav_"))


def ensure_wav(audio_path: Path) -> Path:
    """Convert mp3 to 16kHz mono wav via librosa if needed."""
    if audio_path.suffix.lower() == ".wav":
        return audio_path
    wav_path = TMP_WAV_DIR / f"{audio_path.stem}.wav"
    if not wav_path.exists():
        audio, sr = librosa.load(str(audio_path), sr=16000, mono=True)
        sf.write(str(wav_path), audio, sr)
    return wav_path

In [ ]:
VALID_LABELS = {"Dementia", "Control"}


def classify_audio(wav_path: Path) -> str:
    """Classify a single audio file. Returns raw model response."""
    conversation = [
        {"role": "system", "content": [{"type": "text", "text": SYSTEM_PROMPT}]},
        {
            "role": "user",
            "content": [
                {"type": "audio", "path": str(wav_path)},
                {"type": "text",  "text": USER_PROMPT},
            ],
        },
    ]

    inputs = processor.apply_chat_template(
        conversation,
        tokenize=True,
        add_generation_prompt=True,
        return_dict=True,
    ).to(model.device)

    generated_ids = model.generate(**inputs, max_new_tokens=64)
    output = processor.batch_decode(
        generated_ids[:, inputs.input_ids.shape[1]:],
        skip_special_tokens=True,
    )
    return output[0]

In [ ]:
def evaluate_dataset(csv_path, audio_dir, name=""):
    df = pd.read_csv(csv_path)
    label_map = {0: "Control", 1: "Dementia"}
    predictions, skipped = [], 0

    audio_dir = Path(audio_dir)
    print(f"[{name}] audio_dir={audio_dir}, exists={audio_dir.exists()}")

    for idx, (_, row) in enumerate(tqdm(df.iterrows(), total=len(df), desc=name)):
        label_dir = label_map[row["ad"]]
        matches = list(audio_dir.glob(f"{label_dir}/{row['session_id']}.*"))
        if not matches:
            skipped += 1
            continue
        try:
            raw = classify_audio(ensure_wav(matches[0]))
            cleaned = raw.strip().strip("'\".,;:!?").capitalize()
            pred = cleaned if cleaned in VALID_LABELS else None
        except Exception as e:
            raw, pred = str(e), None
        if idx < 3:
            print(f"  DEBUG [{idx}] session={row['session_id']} raw={repr(raw[:200])} pred={pred}")
        if pred is None:
            print(f"  INVALID [{idx}] session={row['session_id']} true={label_dir} raw={repr(raw[:300])}")
        predictions.append({"session_id": row["session_id"], "true": label_dir, "pred": pred, "raw": raw})

    valid = [p for p in predictions if p["pred"] is not None]
    y_true = [p["true"] for p in valid]
    y_pred = [p["pred"] for p in valid]
    n, total = len(valid), len(df)
    ctrl = [p for p in valid if p["true"] == "Control"]
    dem  = [p for p in valid if p["true"] == "Dementia"]

    print(f"[{name}]")
    print(f"  Accuracy:    {accuracy_score(y_true, y_pred):.4f}")
    print(f"  F1:          {f1_score(y_true, y_pred, pos_label='Dementia'):.4f}")
    print(f"  Control Acc: {sum(p['pred']=='Control'  for p in ctrl)/max(len(ctrl),1):.4f}")
    print(f"  Dementia Acc:{sum(p['pred']=='Dementia' for p in dem) /max(len(dem),1) :.4f}")
    print(f"  Valid: {n}/{total}  Skipped: {skipped}")

In [ ]:
import sys
_train_path = str(PROJECT_ROOT / "train")
if _train_path not in sys.path:
    sys.path.insert(0, _train_path)
from utils.data_split import create_test_csv

csv       = PROJECT_ROOT / "data/processed/Pitt-origin-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Pitt-origin", "Pitt-origin", "Pitt-origin_xlsr_features", xlsr=True)
    
audio_dir = PROJECT_ROOT / "data/raw/Pitt-origin"
evaluate_dataset(csv, audio_dir, "Pitt-origin-raw")

In [ ]:
# csv       = PROJECT_ROOT / "data/processed/Pitt-xlsr-test.csv"
# if not csv.exists() or csv.stat().st_size < 30:
#     create_test_csv(PROJECT_ROOT / "data/raw/Pitt", "Pitt", "Pitt_xlsr_features", xlsr=True)

# audio_dir = PROJECT_ROOT / "data/raw/Pitt"
# evaluate_dataset(csv, audio_dir, "Pitt-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/ADReSS-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/ADReSS", "ADReSS", "ADReSS_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/ADReSS"
evaluate_dataset(csv, audio_dir, "ADReSS-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/ADReSSo-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/ADReSSo", "ADReSSo", "ADReSSo_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/ADReSSo"
evaluate_dataset(csv, audio_dir, "ADReSSo-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/ADReSS-M-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/ADReSS-M", "ADReSS-M", "ADReSS-M_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/ADReSS-M"
evaluate_dataset(csv, audio_dir, "ADReSS-M-raw")

In [ ]:
csv       = PROJECT_ROOT / "data/processed/Lu-xlsr-test.csv"
if not csv.exists() or csv.stat().st_size < 30:
    create_test_csv(PROJECT_ROOT / "data/raw/Lu", "Lu", "Lu_xlsr_features", xlsr=True)

audio_dir = PROJECT_ROOT / "data/raw/Lu"
evaluate_dataset(csv, audio_dir, "Lu-raw")